In [1]:
# ==============================
#  DERMATOLOGY AI PROJECT SETUP
# ==============================

import os
import zipfile
import numpy as np
import tensorflow as tf
from google.colab import drive

print("Mounting Google Drive...")
drive.mount('/content/drive')

# Define project path
PROJECT_PATH = "/content/drive/MyDrive/Dermatology_AI_Project"
DATASET_PATH = f"{PROJECT_PATH}/datasets"

# ZIP file paths
ISIC_ZIP = f"{DATASET_PATH}/ISIC2018_segmentation.zip"
HAM_ZIP = f"{DATASET_PATH}/HAM10000.zip"

print("Project Path:", PROJECT_PATH)
print("Dataset Path:", DATASET_PATH)

# Move to project directory
os.chdir(PROJECT_PATH)
print("Current Working Directory:", os.getcwd())

Mounting Google Drive...
Mounted at /content/drive
Project Path: /content/drive/MyDrive/Dermatology_AI_Project
Dataset Path: /content/drive/MyDrive/Dermatology_AI_Project/datasets
Current Working Directory: /content/drive/MyDrive/Dermatology_AI_Project


In [2]:
!pip install reportlab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 17.4 MB/s eta 0:00:00


In [84]:
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.platypus import Image

In [3]:
import os
import cv2
import numpy as np
import pandas as pd
from datetime import datetime

from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image
from reportlab.lib.styles import getSampleStyleSheet

In [85]:
original_img_dir = "/content/drive/MyDrive/Dermatology_AI_Project/test_images"
mask_dir = "/content/drive/MyDrive/Dermatology_AI_Project/results/severity_analysis/segmentation_masks"
gradcam_dir = "/content/drive/MyDrive/Dermatology_AI_Project/results/gradcam_outputs"

In [112]:
def clinical_interpretation(risk_level):
    if risk_level == "HIGH":
        return "Findings indicate high-risk lesion characteristics."
    elif risk_level == "MEDIUM":
        return "Moderate-risk lesion detected. Clinical evaluation is recommended."
    else:
        return "Low-risk lesion. Regular monitoring is advised."

In [113]:
def generate_clinical_report(image_name, area, diameter, asymmetry,
                            severity_score, risk_level,
                            prediction, confidence):

    interpretation = clinical_interpretation(risk_level)

    report = f"""
===============================
🧾 AI DERMATOLOGY REPORT
===============================

📌 Image ID: {image_name}

-------------------------------
🔍 LESION ANALYSIS
-------------------------------
• Area: {area} pixels
• Diameter: {diameter} px
• Asymmetry Score: {asymmetry:.3f}

-------------------------------
🧠 AI DIAGNOSIS
-------------------------------
• Predicted Condition: {prediction}
• Confidence: {confidence*100:.2f}%

-------------------------------
⚠️ SEVERITY ASSESSMENT
-------------------------------
• Severity Score: {severity_score:.3f}
• Risk Level: {risk_level}

-------------------------------
📋 CLINICAL INTERPRETATION
-------------------------------
• {interpretation}

-------------------------------
💡 RECOMMENDATION
-------------------------------
• Avoid self-medication
• Monitor changes in size/color
• Seek medical advice if symptoms worsen

===============================
"""

    return report

In [114]:
def save_report(report, filepath):
    with open(filepath, "w") as f:
        f.write(report)

In [115]:
# If already in memory:
# results_df = your_dataframe

# OR load from CSV
results_df = pd.read_csv("/content/drive/MyDrive/Dermatology_AI_Project/results/severity_analysis/severity_analysis_results.csv")

results_df.head()

,image,area,diameter,asymmetry,severity_score,risk_level
0,lesion.jpg,1338,58,0.965646,0.712720,HIGH
1,lesion2.jpg,3531,93,0.719706,0.748686,HIGH


In [116]:
from tensorflow.keras.models import load_model

model = load_model(
    "/content/drive/MyDrive/Dermatology_AI_Project/models/mobilenet_finetuned_final.keras",
    compile=False   # ✅ THIS FIXES WARNING
)

In [117]:
class_names = [
    'akiec', 'bcc', 'bkl', 'df',
    'melanoma', 'nv', 'vasc'
]

In [118]:
import cv2

def preprocess_image(img_path):
    img = cv2.imread(img_path)
    img = cv2.resize(img, (224, 224))
    img = img / 255.0
    return np.expand_dims(img, axis=0)

In [119]:
predictions = []
confidences = []

image_dir = "/content/drive/MyDrive/Dermatology_AI_Project/test_images"

for img_name in results_df['image']:

    img_path = os.path.join(image_dir, img_name)

    img = preprocess_image(img_path)
    pred = model.predict(img)[0]

    pred_class = class_names[np.argmax(pred)]
    confidence = np.max(pred)

    predictions.append(pred_class)
    confidences.append(confidence)

# Add to dataframe
results_df['prediction'] = predictions
results_df['confidence'] = confidences

print("✅ Classification added to dataframe!")

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step
✅ Classification added to dataframe!


In [120]:
print(results_df.head())

         image  area  diameter  asymmetry  severity_score risk_level  \
0   lesion.jpg  1338        58   0.965646        0.712720       HIGH   
1  lesion2.jpg  3531        93   0.719706        0.748686       HIGH   

  prediction  confidence  
0         nv    0.979072  
1         nv    0.901074  


In [129]:
row = results_df.iloc[0]

report = generate_clinical_report(
    image_name=row['image'],
    area=row['area'],
    diameter=row['diameter'],
    asymmetry=row['asymmetry'],
    severity_score=row['severity_score'],
    risk_level=row['risk_level'],
    prediction=row['prediction'],
    confidence=row['confidence']
)

print(report)


🧾 AI DERMATOLOGY REPORT

📌 Image ID: lesion.jpg

-------------------------------
🔍 LESION ANALYSIS
-------------------------------
• Area: 1338 pixels
• Diameter: 58 px
• Asymmetry Score: 0.966

-------------------------------
🧠 AI DIAGNOSIS
-------------------------------
• Predicted Condition: nv
• Confidence: 97.91%

-------------------------------
⚠️ SEVERITY ASSESSMENT
-------------------------------
• Severity Score: 0.713
• Risk Level: HIGH

-------------------------------
📋 CLINICAL INTERPRETATION
-------------------------------
• Findings indicate high-risk lesion characteristics.

-------------------------------
💡 RECOMMENDATION
-------------------------------
• Avoid self-medication  
• Monitor changes in size/color  
• Seek medical advice if symptoms worsen  




In [122]:
print(results_df.columns)

Index(['image', 'area', 'diameter', 'asymmetry', 'severity_score',
       'risk_level', 'prediction', 'confidence'],
      dtype='object')


In [130]:
output_dir = "/content/drive/MyDrive/Dermatology_AI_Project/reports/"
os.makedirs(output_dir, exist_ok=True)

for i, row in results_df.iterrows():

    report = generate_clinical_report(
        image_name=row['image'],
        area=row['area'],
        diameter=row['diameter'],
        asymmetry=row['asymmetry'],
        severity_score=row['severity_score'],
        risk_level=row['risk_level'],
        prediction=row['prediction'],
        confidence=row['confidence']
    )

    filename = f"report_{i}.txt"
    filepath = os.path.join(output_dir, filename)

    save_report(report, filepath)

print("✅ All reports generated successfully!")

✅ All reports generated successfully!


In [131]:
results_df['report_text'] = results_df.apply(
    lambda row: generate_clinical_report(
        row['image'],
        row['area'],
        row['diameter'],
        row['asymmetry'],
        row['severity_score'],
        row['risk_level'],
        row['prediction'],
        row['confidence']
    ), axis=1
)

results_df.to_csv("/content/drive/MyDrive/Dermatology_AI_Project/results_with_reports.csv", index=False)

print("✅ Reports embedded in CSV!")

✅ Reports embedded in CSV!


In [132]:
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.cidfonts import UnicodeCIDFont
import os

# ✅ Register font ONCE
pdfmetrics.registerFont(UnicodeCIDFont('HYSMyeongJo-Medium'))

def generate_pdf_report(filepath, image_name, area, diameter, asymmetry,
                        severity_score, risk_level,
                        prediction, confidence):

    doc = SimpleDocTemplate(filepath)
    styles = getSampleStyleSheet()

    # ✅ Apply font
    for style in ['Normal', 'Title', 'Heading2', 'Heading3']:
        styles[style].fontName = 'HYSMyeongJo-Medium'

    content = []

    # -------------------------------
    # Paths (FIXED)
    # -------------------------------
    original_path = os.path.join(original_img_dir, image_name)

    mask_path = os.path.join(
        mask_dir,
        "segmentation_masks",
        image_name
    )

    gradcam_filename = f"gradcam_{image_name}"   # 🔥 FIX
    gradcam_path = os.path.join(gradcam_dir, gradcam_filename)

    # -------------------------------
    # DEBUG (run once if needed)
    # -------------------------------
    print("Original:", original_path, os.path.exists(original_path))
    print("Mask:", mask_path, os.path.exists(mask_path))
    print("GradCAM:", gradcam_path, os.path.exists(gradcam_path))

    # -------------------------------
    # Title
    # -------------------------------
    content.append(Paragraph("🧾 AI Dermatology Clinical Report", styles['Title']))
    content.append(Spacer(1, 20))

    # -------------------------------
    # Images Section
    # -------------------------------
    if os.path.exists(original_path):
        content.append(Paragraph("📷 Original Image", styles['Heading3']))
        content.append(Image(original_path, width=200, height=200))  # smaller = better
        content.append(Spacer(1, 12))

    if os.path.exists(mask_path):
        content.append(Paragraph("🧩 Segmentation Mask", styles['Heading3']))
        content.append(Image(mask_path, width=200, height=200))
        content.append(Spacer(1, 12))

    if os.path.exists(gradcam_path):
        content.append(Paragraph("🔥 Grad-CAM Explanation", styles['Heading3']))
        content.append(Image(gradcam_path, width=200, height=200))
        content.append(Spacer(1, 15))

    # -------------------------------
    # Info Section
    # -------------------------------
    content.append(Paragraph(f"<b>Image ID:</b> {image_name}", styles['Normal']))
    content.append(Spacer(1, 12))

    # -------------------------------
    # Lesion Analysis
    # -------------------------------
    content.append(Paragraph("🔍 Lesion Analysis", styles['Heading2']))
    content.append(Paragraph(f"Area: {area} pixels", styles['Normal']))
    content.append(Paragraph(f"Diameter: {diameter} px", styles['Normal']))
    content.append(Paragraph(f"Asymmetry: {asymmetry:.3f}", styles['Normal']))
    content.append(Spacer(1, 12))

    # -------------------------------
    # Diagnosis
    # -------------------------------
    content.append(Paragraph("🧠 AI Diagnosis", styles['Heading2']))
    content.append(Paragraph(f"Condition: {prediction}", styles['Normal']))
    content.append(Paragraph(f"Confidence: {confidence*100:.2f}%", styles['Normal']))
    content.append(Spacer(1, 12))

    # -------------------------------
    # Severity
    # -------------------------------
    content.append(Paragraph("⚠️ Severity Assessment", styles['Heading2']))
    content.append(Paragraph(f"Severity Score: {severity_score:.3f}", styles['Normal']))

    color = "red" if risk_level == "HIGH" else "orange" if risk_level == "MEDIUM" else "green"
    content.append(Paragraph(
        f"<b>Risk Level:</b> <font color='{color}'>{risk_level}</font>",
        styles['Normal']
    ))
    content.append(Spacer(1, 12))

    # -------------------------------
    # Interpretation
    # -------------------------------
    interpretation = clinical_interpretation(risk_level)
    content.append(Paragraph("📋 Clinical Interpretation", styles['Heading2']))
    content.append(Paragraph(interpretation, styles['Normal']))
    content.append(Spacer(1, 12))

    # -------------------------------
    # Recommendation
    # -------------------------------
    content.append(Paragraph("💡 Recommendation", styles['Heading2']))
    content.append(Paragraph(
        "Consult a certified dermatologist. Avoid self-medication. Monitor lesion changes.",
        styles['Normal']
    ))

    # -------------------------------
    # Build PDF
    # -------------------------------
    doc.build(content)

In [135]:
row = results_df.iloc[0]

pdf_path = "/content/drive/MyDrive/Dermatology_AI_Project/reports/sample_report.pdf"

generate_pdf_report(
    pdf_path,
    row['image'],
    row['area'],
    row['diameter'],
    row['asymmetry'],
    row['severity_score'],
    row['risk_level'],
    row['prediction'],
    row['confidence']
)

print("✅ PDF Generated!")

Original: /content/drive/MyDrive/Dermatology_AI_Project/test_images/lesion.jpg True
Mask: /content/drive/MyDrive/Dermatology_AI_Project/results/severity_analysis/segmentation_masks/segmentation_masks/lesion.jpg False
GradCAM: /content/drive/MyDrive/Dermatology_AI_Project/results/gradcam_outputs/gradcam_lesion.jpg True
✅ PDF Generated!


In [134]:
pdf_dir = "/content/drive/MyDrive/Dermatology_AI_Project/pdf_reports/"
os.makedirs(pdf_dir, exist_ok=True)

for i, row in results_df.iterrows():

    pdf_path = os.path.join(pdf_dir, f"report_{i}.pdf")

    generate_pdf_report(
        pdf_path,
        row['image'],
        row['area'],
        row['diameter'],
        row['asymmetry'],
        row['severity_score'],
        row['risk_level'],
        row['prediction'],
        row['confidence']
    )

print("✅ All PDF reports generated!")

Original: /content/drive/MyDrive/Dermatology_AI_Project/test_images/lesion.jpg True
Mask: /content/drive/MyDrive/Dermatology_AI_Project/results/severity_analysis/segmentation_masks/segmentation_masks/lesion.jpg False
GradCAM: /content/drive/MyDrive/Dermatology_AI_Project/results/gradcam_outputs/gradcam_lesion.jpg True
Original: /content/drive/MyDrive/Dermatology_AI_Project/test_images/lesion2.jpg True
Mask: /content/drive/MyDrive/Dermatology_AI_Project/results/severity_analysis/segmentation_masks/segmentation_masks/lesion2.jpg False
GradCAM: /content/drive/MyDrive/Dermatology_AI_Project/results/gradcam_outputs/gradcam_lesion2.jpg True
✅ All PDF reports generated!


In [108]:
print(os.listdir(gradcam_dir)[:5])

['gradcam_lesion.jpg', 'gradcam_lesion2.jpg']
